# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule.** At the 2026-04-01 origin, review pages with at least 100 feature-window impressions, a known search-position tier, and a CTR at least 20% below that tier's pre-origin pooled CTR. Rank by a transparent opportunity score: relative CTR shortfall × log1p feature impressions × a tier opportunity multiplier. Pages with less evidence remain visible as guarded monitoring/exclusion rows. This is a prioritization rule, not a causal prediction.

Reason codes are `severe_underperformance` (at least 50% below the tier benchmark), `moderate_underperformance` (at least 20% but less than 50% below), `low_volume_watch`, `no_position_data`, and `insufficient_feature_volume`. Only the first two enter the actionable queue; the others remain visible as guarded exclusions.

In [1]:
# Warehouse connection and ML-04 feature/label contract
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your HuggingFace READ token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_retries = 10")
con.execute("SET http_timeout = 120")
con.execute("SET enable_http_metadata_cache = false")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
FEATURE_START, FEATURE_END = "2026-01-01", "2026-03-31"
LABEL_START, LABEL_END = "2026-04-01", "2026-04-30"
MIN_FEATURE_IMPRESSIONS, MIN_LABEL_IMPRESSIONS = 100, 100

# Source and grain guards are intentionally executable but not printed as raw SQL.
source_check = con.sql(f"SELECT COUNT(*) n, MIN(report_date) min_date, MAX(report_date) max_date FROM {FACT}").df()
assert int(source_check.loc[0, "n"]) > 0
assert str(source_check.loc[0, "min_date"])[:10] <= "2025-01-27"
assert str(source_check.loc[0, "max_date"])[:10] >= "2026-06-30"
feature_dupes = con.sql(f"""SELECT COUNT(*) n FROM (SELECT report_date, client_hash_id, content_hash_id FROM {FACT} WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}' GROUP BY 1,2,3 HAVING COUNT(*) > 1)""").df()
assert int(feature_dupes.loc[0, "n"]) == 0

con.sql(f"""CREATE OR REPLACE TEMP VIEW feature_base AS
SELECT f.content_hash_id, f.client_hash_id,
 SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_impressions,0) ELSE 0 END) impressions_90d,
 SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_clicks,0) ELSE 0 END) clicks_90d,
 SUM(CASE WHEN f.gsc_data_available IS TRUE THEN COALESCE(f.gsc_sum_position,0) ELSE 0 END) sum_position_90d,
 SUM(CASE WHEN f.ga4_data_available IS TRUE THEN COALESCE(f.ga4_sessions,0) ELSE 0 END) sessions_90d,
 COUNT(DISTINCT CASE WHEN f.gsc_data_available IS TRUE AND COALESCE(f.gsc_impressions,0)>0 THEN f.report_date END) days_with_impressions_90d,
 ANY_VALUE(c.content_created_date) content_created_date, ANY_VALUE(c.content_type) content_type, ANY_VALUE(c.main_intent) main_intent
FROM {FACT} f LEFT JOIN {DIM_CONTENT} c ON c.content_hash_id=f.content_hash_id
WHERE f.report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
GROUP BY 1,2""")
con.sql(f"""CREATE OR REPLACE TEMP VIEW tier_rates AS
SELECT CASE WHEN impressions_90d=0 OR sum_position_90d=0 THEN 'no_data' WHEN sum_position_90d/impressions_90d<=3 THEN 'top_3' WHEN sum_position_90d/impressions_90d<=10 THEN 'page_1' WHEN sum_position_90d/impressions_90d<=20 THEN 'striking' WHEN sum_position_90d/impressions_90d<=50 THEN 'page_3_5' ELSE 'deep' END position_tier, SUM(clicks_90d)/NULLIF(SUM(impressions_90d),0)*100 expected_ctr_feature FROM feature_base WHERE impressions_90d >= {MIN_FEATURE_IMPRESSIONS} GROUP BY 1""")
feature_vector = con.sql("""SELECT b.*, b.clicks_90d/NULLIF(b.impressions_90d,0)*100 ctr_feature, b.sum_position_90d/NULLIF(b.impressions_90d,0) avg_position_feature, t.position_tier, t.expected_ctr_feature, CASE WHEN t.expected_ctr_feature>0 THEN (b.clicks_90d/NULLIF(b.impressions_90d,0)*100-t.expected_ctr_feature)/t.expected_ctr_feature END relative_ctr_gap, b.sessions_90d>0 has_ga4 FROM feature_base b LEFT JOIN tier_rates t ON t.position_tier=CASE WHEN b.impressions_90d=0 OR b.sum_position_90d=0 THEN 'no_data' WHEN b.sum_position_90d/b.impressions_90d<=3 THEN 'top_3' WHEN b.sum_position_90d/b.impressions_90d<=10 THEN 'page_1' WHEN b.sum_position_90d/b.impressions_90d<=20 THEN 'striking' WHEN b.sum_position_90d/b.impressions_90d<=50 THEN 'page_3_5' ELSE 'deep' END""").df()
assert len(feature_vector)>0 and feature_vector[['content_hash_id','client_hash_id']].duplicated().sum()==0
print(f"Warehouse source rows: {int(source_check.loc[0,'n']):,}; full range {str(source_check.loc[0,'min_date'])[:10]} to {str(source_check.loc[0,'max_date'])[:10]}")
print(f"Feature rows: {len(feature_vector):,}; duplicate feature grains: 0; feature window ends {FEATURE_END}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Warehouse source rows: 78,835,655; full range 2025-01-27 to 2026-06-30
Feature rows: 349,411; duplicate feature grains: 0; feature window ends 2026-03-31


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The queue uses only the ML-04 feature window (2026-01-01 through 2026-03-31). It keeps pseudonymous IDs for grouping and traceability but exports no client names, URLs, queries, product flags, or future metrics. The April future-label query is joined only for evaluation, never for the score.

In [2]:
# Transparent tier-aware score, future-label query for evaluation, and public-safe export
q = feature_vector.copy()
q["feature_eligible"] = q.impressions_90d >= MIN_FEATURE_IMPRESSIONS
q["known_tier"] = q.position_tier.isin(["top_3","page_1","striking","page_3_5","deep"])
q["tier_multiplier"] = q.position_tier.map({"top_3":1.00,"page_1":1.10,"striking":1.20,"page_3_5":1.00,"deep":0.50}).fillna(0.0)
q["shortfall"] = (-q.relative_ctr_gap).clip(lower=0).fillna(0)
q["score"] = np.where(q.feature_eligible & q.known_tier & (q.expected_ctr_feature > 0), q.shortfall * np.log1p(q.impressions_90d) * q.tier_multiplier, 0.0)
q["reason_code"] = np.select([~q.feature_eligible, ~q.known_tier, q.shortfall >= 0.50, q.shortfall >= 0.20], ["insufficient_feature_volume","no_position_data","severe_underperformance","moderate_underperformance"], default="low_volume_watch")
q["action"] = np.where(q.reason_code.isin(["severe_underperformance","moderate_underperformance"]), "review CTR/content fit", "monitor or exclude")
q["confidence_note"] = np.select([q.impressions_90d >= 1000, q.impressions_90d >= 300], ["higher denominator support","moderate denominator support"], default="lower denominator support; verify manually")
q = q.sort_values(["score","impressions_90d"], ascending=[False,False], kind="stable").reset_index(drop=True)
q.insert(0,"rank",np.arange(1,len(q)+1))

# Future labels are isolated to evaluation and use the ML-04 100-impression completeness floor.
label_query = f"""SELECT content_hash_id, client_hash_id, SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END) impressions_next30d, SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END) clicks_next30d FROM {FACT} WHERE report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}' GROUP BY 1,2"""
labels = con.sql(label_query).df()
labels["ctr_next30d"] = labels.clicks_next30d / labels.impressions_next30d.replace(0,np.nan) * 100
eval_q = q.merge(labels, on=["content_hash_id","client_hash_id"], how="left")
eval_q["label_eligible"] = eval_q.impressions_next30d >= MIN_LABEL_IMPRESSIONS
eval_q["need_ctr_fix"] = (eval_q.feature_eligible & eval_q.label_eligible & (eval_q.expected_ctr_feature > 0) & (eval_q.ctr_next30d < eval_q.expected_ctr_feature * 0.5)).astype("Int64")

queue_cols=["rank","content_hash_id","client_hash_id","score","reason_code","action","confidence_note","position_tier","impressions_90d","ctr_feature","expected_ctr_feature","relative_ctr_gap","feature_eligible"]
queue = q[queue_cols].copy()
assert not any(c.lower() in {"need_ctr_fix","ctr_next30d","impressions_next30d","clicks_next30d"} for c in queue.columns)
out=Path("work/outputs"); out.mkdir(parents=True, exist_ok=True)
queue.to_csv(out/"baseline_action_score.csv", index=False)
assert (out/"baseline_action_score.csv").exists()
assert not any(c in queue.columns for c in ["client_name","url","raw_query","product_flag"])
print(f"Ranked queue rows: {len(queue):,}; actionable rows: {int(queue.action.eq('review CTR/content fit').sum()):,}")
print(f"Exported public-safe artifact: {out/'baseline_action_score.csv'}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue rows: 349,411; actionable rows: 78,436
Exported public-safe artifact: work\outputs\baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-20 table is a review aid, not an automatic action list. Each row includes the proposed action, the rule reason, a confidence note based on denominator and tier support, and the main way the rule could be wrong (for example, a volatile low-volume CTR or an unstable position tier).

In [3]:
# Top-20 review with action, reason, confidence, and failure mode
top20 = eval_q.head(20).copy()
top20["failure_mode"] = np.select([top20.impressions_90d < 300, top20.expected_ctr_feature <= 0, top20.ctr_feature <= 0, top20.position_tier.eq("deep")], ["small feature denominator; CTR may be volatile","no positive tier benchmark","zero observed feature CTR; verify tracking or snippet","deep tier position/CTR relationship is weak"], default="position tier may drift before review")
review_cols=["rank","score","action","reason_code","confidence_note","failure_mode","position_tier","impressions_90d","ctr_feature"]
review = top20[review_cols]
assert len(review) == min(20,len(eval_q))
print(review.to_string(index=False))


 rank     score                 action             reason_code            confidence_note                                          failure_mode position_tier  impressions_90d  ctr_feature
    1 12.783715 review CTR/content fit severe_underperformance higher denominator support                 position tier may drift before review      striking          45573.0     0.002194
    2 12.773739 review CTR/content fit severe_underperformance higher denominator support                 position tier may drift before review         top_3         362658.0     0.000827
    3 12.704934 review CTR/content fit severe_underperformance higher denominator support                 position tier may drift before review         top_3         332651.0     0.000301
    4 12.549759 review CTR/content fit severe_underperformance higher denominator support                 position tier may drift before review        page_1         118185.0     0.007615
    5 12.517647 review CTR/content fit severe_underperforman

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks are expected: a high score can still be driven by a small denominator, a stale position aggregate, or a tier with sparse support. I explicitly report weak-pick flags and verify that identifiers are not score inputs, no product flags are used, and feature dates end before the April label window. Future metrics are used only to measure precision@K on eligible rows.

In [4]:
# Weak-pick, leakage, denominator, and precision@K checks
weak = eval_q[(eval_q["rank"] <= 20) & ((eval_q.impressions_90d < 300) | (eval_q.expected_ctr_feature <= 0) | (eval_q.ctr_feature <= 0) | (eval_q.position_tier.eq("deep")))]
print(f"Weak picks in top 20: {len(weak)}")
print(weak[["rank","reason_code","position_tier","impressions_90d","score"]].to_string(index=False))

# Public-safety and leakage assertions. IDs remain grouping/join fields, never score inputs.
score_inputs = {"impressions_90d","relative_ctr_gap","tier_multiplier","shortfall","expected_ctr_feature","position_tier"}
assert not ({"content_hash_id","client_hash_id"} & score_inputs)
for forbidden in ["need_ctr_fix","ctr_next30d","impressions_next30d","clicks_next30d","trend_pct","trend_direction"]:
    assert forbidden not in score_inputs and forbidden not in queue.columns
assert FEATURE_END < LABEL_START
assert (eval_q.loc[eval_q.label_eligible, "impressions_next30d"] >= MIN_LABEL_IMPRESSIONS).all()

def precision_at_k(frame, k):
    eligible = frame[frame.label_eligible & frame.feature_eligible & frame.reason_code.isin(["severe_underperformance","moderate_underperformance"])].sort_values("score", ascending=False)
    return float(eligible.head(k).need_ctr_fix.mean()) if len(eligible.head(k)) else float("nan")
base = float(eval_q.loc[eval_q.label_eligible & eval_q.feature_eligible, "need_ctr_fix"].mean())
metrics = pd.DataFrame({"metric":["eligible_base_rate","precision_at_10","precision_at_20"],"value":[base,precision_at_k(eval_q,10),precision_at_k(eval_q,20)]})
print("Evaluation uses April labels only after the score is frozen; ineligible future denominators are not negatives.")
print(metrics.to_string(index=False))
print("Leakage check: PASS; feature dates end before label start, product/future fields absent, denominator floors enforced.")


Weak picks in top 20: 1
 rank             reason_code position_tier  impressions_90d     score
   18 severe_underperformance         top_3         149311.0 11.913793


Evaluation uses April labels only after the score is frozen; ineligible future denominators are not negatives.
            metric    value
eligible_base_rate 0.585554
   precision_at_10 1.000000
   precision_at_20 1.000000
Leakage check: PASS; feature dates end before label start, product/future fields absent, denominator floors enforced.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.